# Notebook 4: Clickbait Detector Demo
Enter an English headline, choose a trained model, and press **Predict**. The demo displays the predicted class and the model's clickbait confidence.

Run the notebook cells from top to bottom. DistilBERT loads from Google Drive. If the TF-IDF + Logistic Regression `.joblib` file is not already in the runtime, Colab prompts you to upload it.


## 1. Setup

This demo runs in Google Colab. It mounts Drive to load the fine-tuned DistilBERT folder and asks you to upload the TF-IDF model artifact if needed.


In [1]:
%pip -q install transformers==4.50.3 scikit-learn joblib ipywidgets

from pathlib import Path
import html
import joblib
import numpy as np
import torch


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Model locations

The DistilBERT model folder must contain the saved weights, configuration, and tokenizer files. The path below matches the Drive location used by the training notebook for `distilbert_v1`.


In [2]:
from google.colab import drive, files

drive.mount('/content/drive')

DISTILBERT_DIR = Path('/content/drive/MyDrive/clickbait_project/models/distilbert_v1')
TFIDF_LR_PATH = Path('/content/tfidf_lr.joblib')  # Uploaded to this Colab runtime if missing
MAX_LEN = 64


Mounted at /content/drive


## 3. Load the available models

DistilBERT is loaded from Drive if its folder exists. If the TF-IDF artifact is not present in this runtime, Colab asks you to upload one `.joblib` file. The file may contain either a fitted sklearn Pipeline or a dictionary with `vectorizer` and `model` entries.


In [3]:
models = {}

# --- Fine-tuned DistilBERT from Google Drive ---
if DISTILBERT_DIR.is_dir():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    bert_tok = AutoTokenizer.from_pretrained(str(DISTILBERT_DIR), local_files_only=True)
    bert_model = AutoModelForSequenceClassification.from_pretrained(
        str(DISTILBERT_DIR), local_files_only=True
    ).to(device).eval()
    models['DistilBERT'] = 'bert'
    print(f'DistilBERT loaded from {DISTILBERT_DIR} on {device}.')
else:
    print(f'DistilBERT folder not found: {DISTILBERT_DIR}')

# --- TF-IDF + Logistic Regression uploaded as a joblib file ---
if not TFIDF_LR_PATH.exists():
    print('Upload the fitted TF-IDF + Logistic Regression .joblib file.')
    uploaded = files.upload()
    joblib_names = [name for name in uploaded if name.lower().endswith('.joblib')]
    if len(joblib_names) == 1:
        TFIDF_LR_PATH = Path('/content') / joblib_names[0]
    elif len(joblib_names) == 0:
        print('No .joblib file was uploaded; TF-IDF model will be unavailable.')
    else:
        raise ValueError('Upload only one .joblib file per run.')

if TFIDF_LR_PATH.exists():
    tfidf_obj = joblib.load(TFIDF_LR_PATH)
    models['TF-IDF + Logistic Regression'] = 'tfidf'
    print(f'TF-IDF model loaded from {TFIDF_LR_PATH}.')

print('Available models:', list(models))


DistilBERT loaded from /content/drive/MyDrive/clickbait_project/models/distilbert_v1 on cpu.
Upload the fitted TF-IDF + Logistic Regression .joblib file.


Saving tfidf_lr.joblib to tfidf_lr.joblib
TF-IDF model loaded from /content/tfidf_lr.joblib.
Available models: ['DistilBERT', 'TF-IDF + Logistic Regression']


## 4. Prediction functions

Label convention: **0 = non-clickbait, 1 = clickbait**. The displayed percentage is the model's estimated clickbait probability, not a verified severity or misleadingness score.


In [4]:
def clickbait_prob(headline, model_name):
    """Return the model's estimated P(clickbait) in [0, 1]."""
    kind = models[model_name]
    if kind == 'bert':
        enc = bert_tok(
            headline, return_tensors='pt', truncation=True, max_length=MAX_LEN
        ).to(device)
        with torch.no_grad():
            logits = bert_model(**enc).logits
        return torch.softmax(logits, dim=-1)[0, 1].item()

    if isinstance(tfidf_obj, dict):
        features = tfidf_obj['vectorizer'].transform([headline])
        return float(tfidf_obj['model'].predict_proba(features)[0, 1])
    return float(tfidf_obj.predict_proba([headline])[0, 1])


def predict(headline, model_name):
    probability = clickbait_prob(headline, model_name)
    label = 'Clickbait' if probability >= 0.5 else 'Non-clickbait'
    return label, probability


## 5. Demo UI

This classifier was prepared for English headlines. It can accept other languages, but its predictions for them may be unreliable.


In [9]:
import ipywidgets as w
from IPython.display import display, HTML, clear_output

DISCLAIMER = (
    'The percentage is a <b>model confidence estimate</b>, not a verified measure '
    'of how misleading or severe the headline is.'
)

headline_box = w.Textarea(
    placeholder='Enter one English news headline...',
    layout=w.Layout(width='680px', height='80px'),
)
model_pick = w.Dropdown(
    options=list(models),
    description='Model:',
    layout=w.Layout(width='440px'),
    disabled=not models,
)
predict_btn = w.Button(
    description='Predict', button_style='primary', icon='search', disabled=not models
)
out = w.Output()

def on_predict(_):
    with out:
        clear_output()
        text = headline_box.value.strip()
        if not models:
            print('No model loaded. Check the DistilBERT Drive path or upload a .joblib file.')
            return
        if not text:
            print('Please enter a headline first.')
            return
        label, probability = predict(text, model_pick.value)
        color = '#b3261e' if label == 'Clickbait' else '#1b6e3c'
        safe_text = html.escape(text)
        display(HTML(f"""
        <div style='font-family:sans-serif;padding:12px 16px;border-left:5px solid {color};background:#f6f6f6;max-width:680px'>
          <div style='color:#555'>Headline</div>
          <div style='margin-bottom:8px;color:#000000'><i>{safe_text}</i></div>
          <div style='color:#000000'><b>Predicted class:</b> <span style='color:{color};font-weight:bold'>{label}</span></div>
          <div style='color:#000000'><b>Clickbait confidence:</b> {probability * 100:.1f}%</div>
          <div style='color:#555;font-size:0.85em;margin-top:8px'>{DISCLAIMER}</div>
        </div>"""))

predict_btn.on_click(on_predict)
display(w.VBox([headline_box, w.HBox([model_pick, predict_btn]), out]))


### Example headlines

Try one at a time. These are examples only; model predictions can be wrong.
- `You Won't Believe What This Dog Did Next`
- `Central bank raises interest rates by 0.25 percentage points`
- `10 Reasons Your Morning Routine Is Ruining Your Life`
